# 12 — High-wind days

**Purpose.** Define high-wind days from the training distribution and score the forecasts on them:
conditional bias and RMSE, peak attenuation, detection and false-alarm rates, discrimination, and the
days behind the largest errors.

**Answers:** Reviewer 4 comment 12 and the MaxAE part of comment 7; Reviewer 3 (extremes).
**Outputs:** `tables/S12_high_wind_scores.csv`, `tables/S12_largest_errors.csv`. **Runtime:** ~1 min.

In [1]:
# --- Setup: Drive, paths, shared helpers -------------------------------------------------------
import os, sys, json, time, warnings, subprocess
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna', 'shap', 'scienceplots',
                    'ephem', 'catboost', 'lightgbm'], check=False)
except Exception:
    pass
_BASE = os.environ.get('SARKOY_BASE', '/content/drive/MyDrive/windforecast_rev1/')
sys.path.insert(0, os.path.join(_BASE, 'revision'))
if not os.path.exists(os.path.join(_BASE, 'revision', 'revision_utils.py')):
    raise FileNotFoundError('Copy revision_utils.py into ' + os.path.join(_BASE, 'revision') + ' first.')
import revision_utils as ru
import numpy as np, pandas as pd, matplotlib.pyplot as plt
BASE, REV, IN_COLAB, FAST = ru.get_paths()
ru.set_style()
RES = ru.Results(REV)
RES.path = REV + 'results_12.json'
GPU = ru.has_gpu()
try:
    from IPython.display import display
except Exception:
    display = print
print('BASE:', BASE, '| REV:', REV, '| GPU:', GPU, '| FAST (smoke test):', FAST)
print(ru.log_versions(REV))


Mounted at /content/drive
BASE: /content/drive/MyDrive/windforecast_rev1/ | REV: /content/drive/MyDrive/windforecast_rev1/revision/ | GPU: False | FAST (smoke test): False
{'python': '3.13.15', 'numpy': '2.1.3', 'pandas': '2.2.3', 'sklearn': '1.6.1', 'scipy': '1.16.3', 'xgboost': '3.4.1', 'lightgbm': '4.6.0', 'catboost': '1.2.10', 'optuna': '5.0.0', 'shap': '0.52.0', 'statsmodels': '0.15.0', 'matplotlib': '3.10.0', 'seaborn': '0.13.2', 'pyarrow': '23.0.1', 'ephem': '4.2.1', 'tensorflow': '2.20.0', 'gpu': False}


In [2]:
# --- Thresholds from the training period of each outer year ---------------------------------------
df = ru.load_matrix(BASE, REV, 'causal')
y = df[ru.TARGET]; ok = df['target_observed'].astype(bool)
R = pd.read_csv(REV + 'rolling_origin_predictions.csv', parse_dates=['date'])
MODELS = [c for c in ['persistence', 'ElasticNetCV', 'SVR', 'RandomForest', 'LightGBM', 'CatBoost',
                      'XGBoost_selected'] if c in R.columns]
rows, thrs = [], {}
for Y, g in R.groupby('outer_year'):
    train_y = y[(df.index.year < Y) & ok]
    for q, qn in ((0.90, 'P90'), (0.95, 'P95')):
        thr = float(train_y.quantile(q)); thrs[(Y, qn)] = thr
        for m in MODELS:
            rows.append(dict(outer_year=Y, quantile=qn, threshold=thr, model=m, **ru.event_scores(g.y_true, g[m], thr)))
E = pd.DataFrame(rows)
pool = []
for qn in ('P90', 'P95'):
    for m in MODELS:
        yy, pp, ev = [], [], []
        for Y, g in R.groupby('outer_year'):
            thr = thrs[(Y, qn)]
            yy.append(g.y_true.to_numpy()); pp.append(g[m].to_numpy()); ev.append((g.y_true >= thr).to_numpy())
        yy, pp, ev = np.concatenate(yy), np.concatenate(pp), np.concatenate(ev)
        hits = int((ev & (pp >= np.concatenate([np.full(len(g), thrs[(Y, qn)]) for Y, g in R.groupby('outer_year')]))).sum())
        thr_vec = np.concatenate([np.full(len(g), thrs[(Y, qn)]) for Y, g in R.groupby('outer_year')])
        fc = pp >= thr_vec
        h, mi, fa = int((ev & fc).sum()), int((ev & ~fc).sum()), int((~ev & fc).sum())
        from sklearn.metrics import roc_auc_score
        pool.append(dict(quantile=qn, model=m, n_events=int(ev.sum()), hits=h, misses=mi, false_alarms=fa,
                         POD=h / (h + mi) if h + mi else np.nan, FAR=fa / (h + fa) if h + fa else np.nan,
                         CSI=h / (h + mi + fa) if h + mi + fa else np.nan,
                         bias=float(np.mean(pp[ev] - yy[ev])), rmse=ru.rmse(yy[ev], pp[ev]),
                         attenuation=float(np.mean(pp[ev]) / np.mean(yy[ev])),
                         AUC=float(roc_auc_score(ev, pp))))
PL = pd.DataFrame(pool)
pd.concat([E.assign(scope='per_year'), PL.assign(scope='pooled')]).to_csv(REV + 'tables/S12_high_wind_scores.csv', index=False)
display(PL[PL['quantile'] == 'P90'].round(3))
p90 = PL[PL['quantile'] == 'P90'].set_index('model')
RES.put('HW_THR_RANGE', f"{min(v for (Y, q), v in thrs.items() if q == 'P90'):.1f}–{max(v for (Y, q), v in thrs.items() if q == 'P90'):.1f}")
RES.put('HW_N_EVENTS', int(p90.n_events.iloc[0]), 'd')
for m, code in (('XGBoost_selected', 'XGB'), ('persistence', 'PERS')):
    if m in p90.index:
        r = p90.loc[m]
        RES.put(f'HW_BIAS_{code}', r.bias, '+.2f').put(f'HW_RMSE_{code}', r.rmse, '.2f')
        RES.put(f'HW_POD_{code}', 100 * r.POD, '.0f').put(f'HW_FAR_{code}', 100 * r.FAR, '.0f')
        RES.put(f'HW_CSI_{code}', r.CSI, '.2f').put(f'HW_AUC_{code}', r.AUC, '.2f')
        RES.put(f'HW_ATTEN_{code}', 100 * r.attenuation, '.0f')


Loaded causal matrix: (2964, 564), 2016-11-19 -> 2024-12-30, observed targets: 2957


,quantile,model,n_events,hits,misses,false_alarms,POD,FAR,CSI,bias,rmse,attenuation,AUC
0,P90,persistence,152,61,91,91,0.401,0.599,0.251,-1.529,2.385,0.765,0.783
1,P90,ElasticNetCV,152,13,139,6,0.086,0.316,0.082,-2.070,2.333,0.681,0.850
2,P90,SVR,152,18,134,7,0.118,0.280,0.113,-2.065,2.417,0.682,0.848
3,P90,RandomForest,152,28,124,11,0.184,0.282,0.172,-1.753,2.039,0.730,0.875
4,P90,LightGBM,152,30,122,15,0.197,0.333,0.180,-1.692,2.009,0.740,0.879
5,P90,CatBoost,152,31,121,9,0.204,0.225,0.193,-1.762,2.067,0.729,0.876
6,P90,XGBoost_selected,152,28,124,15,0.184,0.349,0.168,-1.727,2.019,0.734,0.877


In [3]:
# --- The days behind the largest errors -----------------------------------------------------------
R['abs_err'] = (R.y_true - R.XGBoost_selected).abs()
R['rise_from_previous_day'] = R.y_true - R.persistence
big = R.nlargest(10, 'abs_err')[['date', 'outer_year', 'y_true', 'XGBoost_selected', 'persistence',
                                 'abs_err', 'rise_from_previous_day']]
big['target_date'] = big.date + pd.Timedelta(days=1)
big.to_csv(REV + 'tables/S12_largest_errors.csv', index=False)
display(big.round(2))
k = int((big.rise_from_previous_day.abs() >= 2).sum())
RES.put('HW_ONSET_SENTENCE', f'{k} of the ten largest errors fell on days when the observed daily mean changed by at '
        f'least 2 m/s from the previous day')
RES.put('HW_MAXERR', float(big.abs_err.max()), '.2f')
RES.put('HW_MAXERR_DATE', big.target_date.iloc[0].strftime('%d %B %Y').lstrip('0'))
RES.save()


,date,outer_year,y_true,XGBoost_selected,persistence,abs_err,rise_from_previous_day,target_date
1129,2023-02-05,2023,12.6,6.99,6.2,5.61,6.4,2023-02-06
1792,2024-11-30,2024,8.8,4.29,5.5,4.51,3.3,2024-12-01
4,2020-01-05,2020,11.2,6.70,3.9,4.50,7.3,2020-01-06
1424,2023-11-28,2023,8.5,4.61,3.2,3.89,5.3,2023-11-29
1465,2024-01-08,2024,7.8,4.02,5.3,3.78,2.5,2024-01-09
1052,2022-11-20,2022,7.6,4.05,3.7,3.55,3.9,2022-11-21
820,2022-04-02,2022,8.0,4.47,6.3,3.53,1.7,2022-04-03
714,2021-12-17,2021,6.5,3.06,3.3,3.44,3.2,2021-12-18
1110,2023-01-17,2023,7.5,4.06,1.6,3.44,5.9,2023-01-18
459,2021-04-06,2021,6.2,2.83,1.7,3.37,4.5,2021-04-07


19 values written to /content/drive/MyDrive/windforecast_rev1/revision/results_12.json
